# Summarize a raw news article with the trained model

Pipeline, the same one the training data went through:

    data/raw_news/{name}.txt ──clean_text──> cleaned article ──BPE──> <s> article </s>
      ──TransformerEncoder (prefix-LM), greedy, one token at a time──> summary </s>

Run this with the project's `.venv` kernel on a machine with a CUDA GPU (the model uses CuPy).
The model needs a few hundred MB of GPU memory; if `train-model` is still running on the same
GPU, wait for it to finish or you may run either process out of memory.

In [2]:
# 1. Imports. The notebook sits in the project root, so `src` is importable from here.
import re
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError(f"Open this notebook from the project root, not {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ProjectConfig
from src.data_preprocessing.batching import SUMMARY_SUFFIX, CreateTrainingBatch
from src.data_preprocessing.text_cleaning_pipeline import clean_text, contains_english, text_to_unicode
from src.inference.greedy_decoder import greedy_summarize
from src.models.encoder import TransformerEncoder
from src.training.train import load_checkpoint

In [3]:
# 2. Tokenizer + model with the same settings as training, then load the trained weights.
config = ProjectConfig.from_yaml(PROJECT_ROOT / "config" / "config.yaml")
RAW_DIR = PROJECT_ROOT / "data" / "raw_news"
SUMMARY_DIR = config.resolve(config.summary_dir)  # mT5 reference summaries, for comparison

# Epoch 8 is the final model (epoch 9 was lost). Set to None to pick the newest summarizer_epoch_N.npz.
CHECKPOINT = config.resolve(config.checkpoint_dir) / "summarizer_epoch_8.npz"
if CHECKPOINT is None:
    # End-of-epoch files only (summarizer_epoch_N.npz), not mid-epoch _step_ or .bak files.
    checkpoints = sorted(
        (p for p in config.resolve(config.checkpoint_dir).glob("summarizer_epoch_*.npz")
         if re.fullmatch(r"summarizer_epoch_\d+", p.stem)),
        key=lambda p: int(p.stem.rsplit("_", 1)[1]),
    )
    if not checkpoints:
        raise FileNotFoundError("No summarizer_epoch_*.npz in checkpoints/; train the model first")
    CHECKPOINT = checkpoints[-1]

vocab_dir = config.resolve(config.vocab_dir)
batcher = CreateTrainingBatch(  # only its tokenizer, special-token IDs and length budgets are used
    batch_size=1,
    vocab_json_data=vocab_dir / "vocab.json",
    merges_txt_data=vocab_dir / "merges.txt",
    max_article_length=config.max_article_length,
    max_summary_length=config.max_summary_length,
)
model = TransformerEncoder(
    vocab_size=batcher.vocab_size,
    d_model=config.d_model,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    d_ff=config.d_ff,
    max_seq_length=config.max_sequence_length,
    tie_weights=config.tie_weights,
)
load_checkpoint(model, CHECKPOINT)
print(f"Loaded {Path(CHECKPOINT).name} | V={batcher.vocab_size} D={config.d_model} N={config.num_layers}")

Loaded summarizer_epoch_8.npz | V=30000 D=512 N=6


In [4]:
# 3. Clean a raw article exactly like scripts/clean_data.py, then summarize it.
def clean_article(path: Path) -> str:
    """Raw file -> cleaned one-line article (UTF-8 fix, NFC, strip HTML, drop ZWJ/ZWNJ, collapse whitespace)."""
    raw_text = text_to_unicode(path)
    if contains_english(raw_text):
        # clean_data.py skips these files, so the model never trained on text like this.
        print(f"Warning: {path.name} contains English letters; training data excluded such articles.")
    cleaned = clean_text(raw_text)
    if not cleaned:
        raise ValueError(f"{path.name} is empty after cleaning")
    return cleaned


def summarize_file(path: str | Path) -> str:
    path = Path(path)
    article = clean_article(path)

    article_tokens = len(batcher.tokenizer.encode(article).ids)
    budget = config.max_article_length - 2  # <s> and </s>
    if article_tokens > budget:
        print(f"Note: article has {article_tokens} tokens; only the first {budget} reach the model.")
    return greedy_summarize(model, batcher, article)

In [5]:
# 4. Pick any file from data/raw_news and summarize it.
ARTICLE = RAW_DIR / "100001.txt"

summary = summarize_file(ARTICLE)

print("Article:\n", clean_article(ARTICLE)[:1000], "...\n")
print("Model summary:\n", summary, "\n")

reference = SUMMARY_DIR / f"{ARTICLE.stem}{SUMMARY_SUFFIX}"
if reference.is_file():
    print("reference summary (training target):\n", reference.read_text(encoding="utf-8"))

Note: article has 2046 tokens; only the first 766 reach the model.
Article:
 २६ असार, काठमाडौं । सर्वोच्च अदालतले अनुसन्धानका सिलसिलामा आरोपितहरुको बयान गर्दा अडियो भिजुवल रेकर्डिङ प्रणाली लागू गर्न सरकारका नाममा निर्देशनात्मक आदेश जारी गरेको छ । सर्वोच्चले सम्भव भएसम्म सबैजसो अनुसन्धानमा र नभए तत्कालका लागि गम्भीर प्रकृतिका फौजदारी मुद्दाहरूमा अडियो भिजुअल रेकर्डिङमा मात्रै बयान गर्न निर्देशन दिएको हो । सर्वोच्च अदालतले त्यसका लागि भौतिक पूर्वाधार, प्रविधि र बजेटको व्यवस्था गर्नु भनेको छ । सर्वोच्चले अनुसन्धानका क्रममा आरोपितहरूमाथि हुनसक्ने शारीरिक तथा मानसिक यातनाका गुनासाहरूलाई सधैंका लागि निरुत्साहित गर्न र अनुसन्धानको वैधता पुष्टि गर्न भन्दै यस्तो आदेश दिएको हो । न्यायाधीशहरू हरि फुयाल र बालकृष्ण ढकालको इजलासले पक्राउ परेका आरोपितहरूले बयानका क्रममा आफ्ना कानून व्यवसायीको भौतिक उपस्थिति माग गरेमा उपलब्ध गराउनु भनी आदेश दिएको छ । सर्वोच्चको आदेश अनुसार, त्यसरी उपस्थित हुने कानून व्यवसायीले ‘अनुसन्धानको गोपनीयता र प्रक्रियामा अवरोध नहुने’ गरी बयान कक्षमा पहुँच पाउनेछन् । उसले कानून

In [6]:
# 5. Or summarize text you paste in directly (it goes through the same cleaning).
TEXT = """
१३ असोज, काठमाडौं । तीन दिनको लगातारको झरी र प्राकृतिक विपत्तिका कारण उत्तरी छिमेकी मुलुक चीनसँगको स्थलगत व्यापार पूर्णरूपमा ठप्प भएको छ ।

दसैं आउन दुई साता मात्र बाँकी रहँदा चीनसँग जोडिएका प्रमुख तीनवटै व्यापारिक नाका एकपछि अर्को गर्दै बन्द भएपछि दुई देशबीचको स्थलमार्गबाट हुने आपूर्ति शृंखला टुटेको छ ।




चालु आर्थिक वर्षको पहिलो दुई महिनामै नेपालले चीनबाट ७६ अर्ब ४० करोड रुपैयाँभन्दा बढीको वस्तु आयात गरेको भन्सार विभागको तथ्यांक छ । तर, अब प्रमुख तीनवटै नाका जाने सडक र संरचना भत्किएपछि स्थलमार्गबाट हुने ठूलो परिमाणको व्यापारिक कारोबार असम्भव बनेको छ ।

पहिरोको जोखिम देखाउँदै चीनले सबैभन्दा पुरानो तातोपानी–खासा नाका यसअघि नै बन्द गरेको थियो । त्यसपछि १० भदौमा भोटेकोशीमा आएको बाढीले रसुवागढी नाकाको भौतिक संरचना पूर्णरूपमा ध्वस्त बनायो ।




यी दुई मुख्य नाका बन्द भएपछि नेपाल–चीन व्यापारको भरपर्दो विकल्प बनेको मुस्ताङको कोरला नाका पनि पछिल्लो बाढीका कारण सडक सम्पर्कबाट विच्छेद भएको छ।

कालीगण्डकी नदीमा आएको बाढीले कालीगण्डकी करिडोर र मध्यपहाडी लोकमार्ग अन्तर्गत पर्ने महत्त्वपूर्ण पुलका एप्रोच सडक बगाएपछि कोरला नाका पुग्ने बाटो पूर्णरूपमा अवरुद्ध भएको हो । कोरला नाका र त्यहाँको भन्सार कार्यालय सुरक्षित भए पनि बेनी–जोमसोम सडक खण्डका पुलमा क्षति पुग्दा मालवाहक सवारीसाधनको आवागमन रोकिएको छ ।

सडकमै रोकियो अर्बौंको व्यापार

तातोपानी र रसुवागढी बन्द भएपछि कोरला नाकामा व्यापारिक चाप ह्वात्तै बढेको थियो । १० देखि २५ भदौसम्मको १५ दिनमा मात्रै यो नाकाबाट ४ सय ३४ कन्टेनर र ६ सय ४१ वटा विद्युतीय सवारीसाधन नेपाल भित्रिएका थिए । चालु आर्थिक वर्षको भदौ अन्तिम सातासम्म मात्रै मुस्ताङ भन्सारले २ अर्ब ३१ करोड रुपैयाँभन्दा बढी राजस्व संकलन गरिसकेको थियो ।

अघिल्लो वर्ष १४ अर्ब ४१ करोड बराबरको आयात र ३ हजारभन्दा बढी विद्युतीय सवारी भित्र्याएको यो नाकाको बाटो नै काटिएपछि अब अर्बौंको व्यापार सडकमै रोकिएको छ ।

काठमाडौंमा जम्मा ३० प्रतिशत सामान, मूल्य ३० प्रतिशत बढ्ने

नाका बन्द हुँदा चीनबाट भित्रिने कपडा, जुत्ताचप्पल, झोलादेखि इलेक्ट्रोनिक्स र विद्युतीय गाडीसम्मको ढुवानी रोकिएको छ, जसले व्यापारी र उपभोक्ता दुवैलाई चिन्तित बनाएको छ ।

आरबी कम्प्लेक्स व्यवसायी संघका अध्यक्ष परशुराम दाहालका अनुसार अहिलेसम्म दसैंका लागि ल्याइएका जम्मा २० देखि ३० प्रतिशत मात्र सामान काठमाडौं आइपुगेको छ। ‘दसैँ लक्षित सामान बाटोमै रोकिएका छन्, गोदाममा भएका सामान बाटो डाइभर्ट गरेर कोरला नाकातिरबाट ल्याउन खोजिएको थियो, तर त्यो पनि सबै रोकियो,’ अध्यक्ष दाहाल भन्छन्, ‘अहिलेसम्म जम्मा बढीमा ३० प्रतिशत मात्र सामान आइपुगेको छ, त्योभन्दा बढी आएको छैन ।’

सामान आयातमा भन्सारको तर्फबाट खासै झन्झट नरहेको भए पनि डलरको भाउ र ढुवानी भाडा बढेका कारण मूल्य महँगिने अवस्था आएको उनले बताए । ‘डलरको भाउ पनि एकदमै बढेको छ, बाटो बन्द भएपछि ढुवानी भाडा पनि बढाइएको छ,’ दाहालले भने, ‘भाडा बढेका कारण अब आउने सामानको मूल्य २५ देखि ३० प्रतिशतसम्म महँगो पर्ने देखिन्छ ।’

दाहालका अनुसार ठूला मालबाहक गाडी चल्न नसकेपछि कतिपय व्यवसायीले भारतबाट साना गाडीमा सामान ल्याएर व्यापार गरिरहेका छन् । तर, भारतीय सामान चिनियाँभन्दा महँगो पर्ने र साना गाडीको ढुवानी खर्च बढी लाग्ने उनी बताउँछन् । चीनबाट सामान नआए पनि बजारमा पुरानो स्टक र स्वदेशी (लोकल) उत्पादनले केही राहत दिएको छ ।

यसका साथै, लगातार ४–५ दिन पानी पर्दा बाटोघाटो बिग्रिएर बाहिरी जिल्लाका व्यापारी सामान लिन काठमाडौं नआउँदा होलसेल व्यापार ठप्प जस्तै बनेको छ ।

कतिपयले ह्वाट्सएप र सामाजिक सञ्जालबाटै अर्डर गरिरहे पनि धेरै व्यापार नभएको दाहाल बताउँछन् । ‘लगातार तीन वर्ष भइसक्यो, असोजमै पानी परेर बाटोघाटो भत्किने र व्यापार बिग्रिने गरेको छ,’ दाहालले गुनासो गरे, ‘व्यापारीलाई सधैं मार पर्ने भयो, सबैले यो विपत्ति सहनै पर्ने अवस्था छ ।’

पूर्वाधार र कूटनीतिक पहलको अभावले व्यवसायी पीडामा

नेपाल राष्ट्रिय व्यवसायी महासंघका अध्यक्ष मनोजबाबु श्रेष्ठ चीनतर्फका तीनवटै नाका बन्द हुँदा व्यवसायी, बजार र आपूर्ति व्यवस्था नराम्ररी प्रभावित भएको बताउँछन् ।

सरकारले कूटनीतिक पहल नगर्दा र चीनले विभिन्न बाहना बनाउँदा अर्बौंको लगानीमा तयार भएका नाकाहरू प्रयोगविहीन भएको उनको भनाइ छ ।

भूकम्पपछि तातोपानी नाकामा अर्बौं रुपैयाँ खर्च गरेर ड्राइपोर्ट (सुक्खा बन्दरगाह) र पार्किङलगायतका सम्पूर्ण सुविधासम्पन्न भौतिक संरचना बनिसकेको छ । तर, चिनियाँ पक्षले आन्तरिक कारण देखाएर खुलाउन नचाहेको र नेपाल सरकारले ठोस पहल नगरेको उनी बताउँछन् ।

‘हामीले कूटनीतिक पहल गरेर तातोपानी नाका तुरुन्त खुलाउनुपर्छ भनेर सरकारलाई भनिसकेका छौं, हिजो संसदमा पनि सांसदहरूले यो कुरा उठाएका छन्,’ श्रेष्ठले भने, ‘तर, सरकारले किन नाका खुलाउन चाहिरहेको छैन भन्ने बुझ्न सकिएको छैन।’

कोरला नाकामा पनि भन्सार पासमा ढिलासुस्ती र पूर्वाधारको अभाव छ । सुरुवाती १५ दिनमा दैनिक ५० वटा कन्टेनरको भन्सार जाँचपास हुने गरेको भए पनि अहिले काम सुस्त भएको श्रेष्ठले बताए ।

‘कोरला नाकामा सरकारको भौतिक संरचना राम्रो छैन, जनशक्ति पनि कम छ,’ उनले भने, ‘त्यहाँ सफ्टवेयर र नेटवर्किङको ठूलो समस्या छ, कहिले सर्भर डाउन हुने, कहिले नेटवर्क नआउने समस्या एक वर्षदेखि चलिरहेको छ, जसलाई सरकारले सुधार्न सकेको छैन ।’

भौगोलिक दूरीका कारण कोरला नाकाबाट सामान ल्याउँदा ढुवानी भाडा अत्यधिक महँगो पर्न गएको छ । केरुङबाट कोरला पुग्न मात्रै चीनतर्फ १२ सय किलोमिटर घुम्नुपर्ने र कोरलाबाट यता २०० किलोमिटर अत्यन्तै बिग्रिएको बाटो आउनुपर्ने अवस्था छ ।

‘ट्रान्सपोर्टेसनका कारण एक जोर जुत्ता वा ज्याकेटमा मात्रै सयदेखि डेढ सय रुपैयाँसम्म महँगो पर्छ,’ श्रेष्ठले भने, ‘सामान महँगो भए पनि बजारमा सहज आपूर्ति हुन्थ्यो, तर अहिले सबै ब्लक भएको छ ।’

नाका बन्दले बजारको अवस्था असन्तुलित बनेको उनी बताउँछन् । ‘व्यवसायीहरू अत्यन्तै पीडामा छन्, दसैं अगाडि बजारले जुन पिकअप लिनुपर्ने हो र जुन उत्साह देखिनुपर्ने हो, त्यो पटक्कै छैन,’ उनले थपे ।

के छ नाकाको ग्राउन्ड रियालिटी ?

मुस्ताङ भन्सार कार्यालयका प्रमुख भन्सार अधिकृत विदुर चुडालका अनुसार अहिले भन्सार पास एकासहित डेढ सयभन्दा बढी मालवाहक गाडी र कन्टेनर भन्सारमै रोकिएका छन् ।

यार्डमा र नाकामा गरेर ३ देखि ४ सय ईभी गाडी र खाली तथा भरिएका कन्टेनरहरू थन्किएका छन् । ‘चीनतर्फबाट गाडीहरूको फ्लो बढे पनि नेपालतर्फको बाटो बन्द भएकाले व्यापारीले भन्सार जाँचपासमा हतार गरेका छैनन्,’ भन्सार प्रमुख चुडालले भने ।

बाटो मर्मतका लागि सडक विभागले तीव्र गतिमा काम गरिरहेको र बाटो खुल्ने बित्तिकै सहजरूपमा सामान काठमाडौं आइपुग्ने उनले बताए ।
"""

print(greedy_summarize(model, batcher, clean_text(TEXT)))

नेपाल–चीन व्यापारको भरपर्दो विकल्प बनेको मुस्ताङको कोरला नाकारोबार सडक सम्पर्कबाट विच्छेद भएको छ । पहिरोको जोखिम देखाउँदै चीनले सबैभन्दा पुरानो तातोपानी–खासा नाका यसअघि नै बन्द गरेको थियो ।
